# Filtro de 1 pasada (screen) contra la búsqueda real, Colab L4

El filtro `screen_patch_candidates` acertó en el Qwen3.5-0.8B, pero ahí ya sabíamos la respuesta. Aquí se prueba **a ciegas** en otro modelo: la regla (pareto_score, umbral dNLL 0.05, 6 secuencias apartadas) quedó fija en GitHub **antes** de ver los resultados del 4B.

**Predicción:** la configuración que eligió la búsqueda real (generando respuestas) queda en el **top 3 de 12** del filtro. Al azar: 25%.

**Necesita:** la carpeta de Drive `wsi_runs/discover_fast_qwen35-4b` de la corrida rápida (con `diagnosis_blind.json`). Solo la lee; trabaja en una copia.

**Uso:** `L4` → `Run all` → acepta Drive. ~10–20 min (casi todo es bajar el modelo). Resultado en `MyDrive/wsi_runs/FINALES/screen_fast_<modelo>.json`; el runtime se apaga solo si ese archivo existe.

**No lo corras en el mismo runtime que la corrida rápida si sigue viva.**

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"
TAG = MODEL.split("/")[-1].lower().replace(".", "")
RUN = f"discover_fast_{TAG}"               # carpeta de la corrida rápida en Drive
RUN_NAME = f"screen_fast_{TAG}"
REPO = "/content/wsi"
BASE = "/content/base"                     # misma ruta que usó la corrida rápida (config.json la guarda)
W = "/content/work_screen"
S = f"{REPO}/scripts/screen_fast_grid.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers
!pip install -q flash-linear-attention || echo "fla opcional"
import transformers; print("transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
SRC = f"/content/drive/MyDrive/wsi_runs/{RUN}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(FINALES, exist_ok=True); os.makedirs(W, exist_ok=True)
need = ["config.json", "requests_elicit.jsonl", "outputs_elicit.jsonl", "diagnosis_blind.json", "blind_directions.npz"]
for f in need + ["search.json"]:
    if os.path.exists(f"{SRC}/{f}"):
        shutil.copy2(f"{SRC}/{f}", W)         # solo lectura de la carpeta original
missing = [f for f in need if not os.path.exists(f"{W}/{f}")]
assert not missing, f"faltan en {SRC}: {missing} (¿ya terminó el diagnóstico de la corrida rápida?)"
d = json.load(open(f"{W}/diagnosis_blind.json"))
assert d.get("diag_version") == 2, "diagnóstico viejo (v1): espera a que la corrida rápida rehaga el diagnóstico"
print("search.json:", "sí (se califica al final)" if os.path.exists(f"{W}/search.json") else "todavía no (solo ranking)")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
assert os.path.exists(S), f"{S} no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
# El filtro: 12 parches (los mismos de la búsqueda) evaluados con 1 pasada cada uno. Primero guarda el ranking,
# después abre search.json y califica.
r = subprocess.run([sys.executable, "-u", S, "--work", W], capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-4000:])
assert r.returncode == 0, "el filtro falló: mira el error arriba"

In [ ]:
import datetime
res = json.load(open(f"{W}/results_screen_fast.json"))
res["_final"] = {"complete": "vs_search" in res, "model": MODEL, "commit": COMMIT, "notebook": "screen_fast_colab.ipynb",
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("guardado en", dst)
if "vs_search" in res:
    c = res["vs_search"].get("choice", {})
    print(f"elección de la búsqueda: {c.get('name')} -> lugar {c.get('screen_rank')} de 12 en el filtro;",
          "ACIERTO (top 3)" if c.get("in_top3") else "FALLO (fuera del top 3)")
else:
    print("La búsqueda no ha terminado: el ranking quedó guardado. Vuelve a correr cuando exista search.json.")

In [ ]:
# Apagar el runtime: solo si el resultado calificado ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: falta el resultado calificado en FINALES.")